
# VWorld 노인복지시설 API 수집 — 서울·경기·전라도

VWorld 2D 데이터 API의 **노인복지시설 (`LT_P_MGPRTFB`)** 데이터를 수집합니다.

이 노트북은 다음을 수행합니다.

- 서울특별시 노인복지시설 수집 → `df_seoul`
- 경기도 노인복지시설 수집 → `df_gyeonggi`
- 전북특별자치도 노인복지시설 수집 → `df_jeonbuk`
- 전라남도 노인복지시설 수집 → `df_jeonnam`
- 전라도(전북+전남) 통합 → `df_jeolla`
- 전체(서울+경기+전북+전남) 통합 → `df_all`
- 각 DataFrame을 CSV / JSON으로 저장
- API 원본 GeoJSON 응답도 지역별 JSON으로 저장

> **중요:** 경기도 BBOX 안에는 서울이 포함되므로, 단순 BBOX 호출만 사용하지 않습니다.  
> 먼저 VWorld의 시도 경계(`LT_C_ADSIDO_INFO`)를 가져온 뒤, 시설 좌표가 실제 해당 시도 polygon 내부에 있는지 공간 필터링합니다.

### 준비
1. VWorld API 인증키를 발급합니다.
2. 아래 설정 셀의 `VWORLD_API_KEY`에 키를 입력합니다.
3. 키 발급 시 등록한 도메인이 있다면 `VWORLD_DOMAIN`도 맞춰 주세요.

참고 링크: https://www.vworld.kr/dev/v4dv_2ddataguide2_s002.do?svcIde=mgprtfb


In [1]:

# 필요한 패키지 설치
%pip install -q requests pandas shapely


Note: you may need to restart the kernel to use updated packages.


c:\Users\User\OneDrive\Desktop\SKN 부트캠프\3rd-project\.venv\Scripts\python.exe: No module named pip


In [1]:

import os
import json
import time
from pathlib import Path

import requests
import pandas as pd
from shapely.geometry import shape, Point

pd.set_option("display.max_columns", None)

# ============================================================
# 1. 사용자 설정
# ============================================================


VWORLD_API_KEY = os.getenv("VWORLD_API_KEY", "").strip()

# VWorld 키 발급 시 등록한 도메인에 맞게 수정하세요.
# 로컬/Jupyter에서 테스트할 때 등록한 도메인이 http://localhost 라면 그대로 사용합니다.
VWORLD_DOMAIN = os.getenv("VWORLD_DOMAIN", "http://localhost").strip()

BASE_URL = "https://api.vworld.kr/req/data"

# 노인복지시설 서비스 ID
FACILITY_SERVICE_ID = "LT_P_MGPRTFB"

# 광역시도 경계 서비스 ID
SIDO_SERVICE_ID = "LT_C_ADSIDO_INFO"

# 한 페이지 요청 건수
PAGE_SIZE = 1000

# API 과도 호출 방지
REQUEST_INTERVAL_SEC = 0.25

# 결과 저장 폴더
OUTPUT_DIR = Path("vworld_elderly_welfare_output")
OUTPUT_DIR.mkdir(exist_ok=True)

if not VWORLD_API_KEY:
    raise ValueError(
        "VWORLD_API_KEY가 비어 있습니다. "
        "이 셀의 VWORLD_API_KEY 변수에 인증키를 입력한 뒤 다시 실행하세요."
    )

print("설정 완료")
print("저장 폴더:", OUTPUT_DIR.resolve())


설정 완료
저장 폴더: C:\Users\User\OneDrive\Desktop\SKN 부트캠프\3rd-project\src\vworld_elderly_welfare_output



## 2. VWorld API 호출 함수

VWorld 2D 데이터 API의 `GetFeature` 요청을 페이지 단위로 반복 호출합니다.

응답은 일반적으로 다음 구조를 사용합니다.

`response → result → featureCollection → features`

`NOT_FOUND`는 해당 페이지에 더 이상 데이터가 없는 경우로 처리합니다.


In [2]:

def vworld_get_feature(
    service_id,
    page=1,
    size=PAGE_SIZE,
    geom_filter=None,
    geometry=True,
    attribute=True,
    crs="EPSG:4326",
):
    """VWorld 2D Data API GetFeature 1페이지 호출"""

    params = {
        "service": "data",
        "version": "2.0",
        "request": "GetFeature",
        "format": "json",
        "errorformat": "json",
        "size": size,
        "page": page,
        "data": service_id,
        "geometry": str(geometry).lower(),
        "attribute": str(attribute).lower(),
        "crs": crs,
        "key": VWORLD_API_KEY,
        "domain": VWORLD_DOMAIN,
    }

    if geom_filter:
        params["geomFilter"] = geom_filter

    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()

    try:
        data = response.json()
    except Exception as e:
        raise RuntimeError(
            f"JSON 파싱 실패\nURL: {response.url}\n"
            f"응답 앞부분: {response.text[:500]}"
        ) from e

    api_response = data.get("response", {})
    status = api_response.get("status")

    if status == "ERROR":
        error = api_response.get("error", {})
        raise RuntimeError(
            f"VWorld API ERROR | code={error.get('code')} | "
            f"text={error.get('text')}\n요청 URL: {response.url}"
        )

    if status == "NOT_FOUND":
        return [], data

    feature_collection = (
        api_response
        .get("result", {})
        .get("featureCollection", {})
    )

    features = feature_collection.get("features", []) or []
    return features, data


def fetch_all_features(
    service_id,
    geom_filter=None,
    page_size=PAGE_SIZE,
    max_pages=1000,
    verbose=True,
):
    """페이지를 끝까지 순회하며 모든 feature를 수집"""

    all_features = []
    raw_pages = []

    for page in range(1, max_pages + 1):
        features, raw = vworld_get_feature(
            service_id=service_id,
            page=page,
            size=page_size,
            geom_filter=geom_filter,
            geometry=True,
            attribute=True,
            crs="EPSG:4326",
        )

        raw_pages.append(raw)

        if verbose:
            print(f"page {page}: {len(features):,}건")

        if not features:
            break

        all_features.extend(features)

        # 마지막 페이지 판단
        if len(features) < page_size:
            break

        time.sleep(REQUEST_INTERVAL_SEC)

    return all_features, raw_pages



## 3. 대한민국 광역시도 경계 불러오기

전국 BBOX로 광역시도 polygon을 한 번에 가져온 뒤, 속성값에서 지역명을 찾아 매칭합니다.

전북은 데이터 버전에 따라 `전북특별자치도` 또는 과거 명칭 `전라북도`가 들어 있을 수 있어 둘 다 인식하도록 처리합니다.


In [3]:

KOREA_BBOX = "BOX(124,33,132,39.5)"

sido_features, sido_raw_pages = fetch_all_features(
    SIDO_SERVICE_ID,
    geom_filter=KOREA_BBOX,
    verbose=True,
)

print("\n광역시도 feature 수:", len(sido_features))

# 어떤 속성 컬럼이 들어왔는지 확인
if sido_features:
    print("\n첫 번째 feature properties:")
    print(sido_features[0].get("properties", {}))


page 1: 16건

광역시도 feature 수: 16

첫 번째 feature properties:
{'ctp_eng_nm': 'Daegu', 'ctprvn_cd': '27', 'ctp_kor_nm': '대구광역시'}


In [4]:

def feature_property_values(feature):
    props = feature.get("properties") or {}
    return [str(v).strip() for v in props.values() if v is not None]


def find_sido_feature(features, aliases):
    """properties 값 중 alias와 정확히 일치하는 광역시도 feature 탐색"""
    aliases = [a.strip() for a in aliases]

    # 1차: 정확 일치
    for feature in features:
        values = feature_property_values(feature)
        if any(alias in values for alias in aliases):
            return feature

    # 2차: 부분 일치
    for feature in features:
        values = feature_property_values(feature)
        joined = " | ".join(values)
        if any(alias in joined for alias in aliases):
            return feature

    raise ValueError(
        f"광역시도 경계를 찾지 못했습니다. aliases={aliases}\n"
        "바로 위 셀에서 sido_features의 properties를 확인하세요."
    )


REGION_SPECS = {
    "seoul": {
        "label": "서울특별시",
        "aliases": ["서울특별시", "서울"],
    },
    "gyeonggi": {
        "label": "경기도",
        "aliases": ["경기도", "경기"],
    },
    "jeonbuk": {
        "label": "전북특별자치도",
        "aliases": ["전북특별자치도", "전라북도", "전북"],
    },
    "jeonnam": {
        "label": "전라남도",
        "aliases": ["전라남도", "전남"],
    },
}

region_polygons = {}
region_boundary_features = {}

for key, spec in REGION_SPECS.items():
    feature = find_sido_feature(sido_features, spec["aliases"])
    geom = feature.get("geometry")

    if not geom:
        raise ValueError(f"{spec['label']} 경계 geometry가 없습니다.")

    polygon = shape(geom)

    region_boundary_features[key] = feature
    region_polygons[key] = polygon

    minx, miny, maxx, maxy = polygon.bounds
    print(
        f"{key:10s} | {spec['label']:10s} | "
        f"bounds=({minx:.5f}, {miny:.5f}, {maxx:.5f}, {maxy:.5f})"
    )


seoul      | 서울특별시      | bounds=(126.76448, 37.42830, 127.18379, 37.70146)
gyeonggi   | 경기도        | bounds=(126.38010, 36.89378, 127.84811, 38.28226)
jeonbuk    | 전북특별자치도    | bounds=(125.96690, 35.29923, 127.91137, 36.15657)
jeonnam    | 전라남도       | bounds=(125.07773, 33.96857, 127.90563, 35.48691)



## 4. 지역별 노인복지시설 수집

각 시도의 실제 polygon 경계에서 BBOX를 만들고 시설을 조회한 후,
시설 좌표가 polygon 안에 있는 경우만 남깁니다.

따라서 **서울이 경기도 데이터에 섞이는 문제를 방지**할 수 있습니다.


In [5]:

def polygon_to_bbox_filter(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    return f"BOX({minx},{miny},{maxx},{maxy})"


def feature_point(feature):
    """Point feature를 shapely Point로 변환. Point가 아니면 대표점을 사용."""
    geom = feature.get("geometry")
    if not geom:
        return None

    try:
        g = shape(geom)
    except Exception:
        return None

    if g.is_empty:
        return None

    if g.geom_type == "Point":
        return g

    return g.representative_point()


def filter_features_by_polygon(features, polygon):
    kept = []

    for feature in features:
        pt = feature_point(feature)
        if pt is None:
            continue

        # 경계선 위 점도 포함하기 위해 covers 사용
        if polygon.covers(pt):
            kept.append(feature)

    return kept


def fetch_region_facilities(region_key):
    spec = REGION_SPECS[region_key]
    polygon = region_polygons[region_key]
    bbox_filter = polygon_to_bbox_filter(polygon)

    print("=" * 70)
    print(f"{spec['label']} 조회")
    print("geomFilter:", bbox_filter)

    bbox_features, raw_pages = fetch_all_features(
        FACILITY_SERVICE_ID,
        geom_filter=bbox_filter,
        verbose=True,
    )

    filtered = filter_features_by_polygon(bbox_features, polygon)

    print(
        f"→ BBOX 조회: {len(bbox_features):,}건 / "
        f"실제 {spec['label']} 경계 내부: {len(filtered):,}건"
    )

    return filtered, raw_pages


region_features = {}
region_raw_pages = {}

for region_key in REGION_SPECS:
    features, raw_pages = fetch_region_facilities(region_key)
    region_features[region_key] = features
    region_raw_pages[region_key] = raw_pages
    time.sleep(REQUEST_INTERVAL_SEC)


서울특별시 조회
geomFilter: BOX(126.76448395976817,37.42829751474203,127.18379492560015,37.70145532189244)
page 1: 6건
→ BBOX 조회: 6건 / 실제 서울특별시 경계 내부: 4건
경기도 조회
geomFilter: BOX(126.38010162310795,36.893775053414146,127.84811314551624,38.28226058663642)
page 1: 75건
→ BBOX 조회: 75건 / 실제 경기도 경계 내부: 43건
전북특별자치도 조회
geomFilter: BOX(125.96689745878338,35.29923263222256,127.91136737991542,36.15657046920477)
page 1: 69건
→ BBOX 조회: 69건 / 실제 전북특별자치도 경계 내부: 57건
전라남도 조회
geomFilter: BOX(125.0777266852185,33.96857281719295,127.90562660392531,35.4869137796544)
page 1: 74건
→ BBOX 조회: 74건 / 실제 전라남도 경계 내부: 60건



## 5. GeoJSON feature → pandas DataFrame 변환

`properties`의 모든 속성을 그대로 열로 펼치고,
geometry에서 `longitude`, `latitude`를 별도 열로 추가합니다.

VWorld 원본에 어떤 속성 컬럼이 들어오더라도 최대한 그대로 보존하는 방식입니다.


In [6]:

def features_to_dataframe(features, region_label):
    rows = []

    for feature in features:
        row = dict(feature.get("properties") or {})

        row["_feature_id"] = feature.get("id")
        row["_geometry_type"] = (feature.get("geometry") or {}).get("type")
        row["region"] = region_label

        pt = feature_point(feature)
        if pt is not None:
            row["longitude"] = pt.x
            row["latitude"] = pt.y
        else:
            row["longitude"] = None
            row["latitude"] = None

        rows.append(row)

    return pd.DataFrame(rows)


df_seoul = features_to_dataframe(
    region_features["seoul"],
    REGION_SPECS["seoul"]["label"],
)

df_gyeonggi = features_to_dataframe(
    region_features["gyeonggi"],
    REGION_SPECS["gyeonggi"]["label"],
)

df_jeonbuk = features_to_dataframe(
    region_features["jeonbuk"],
    REGION_SPECS["jeonbuk"]["label"],
)

df_jeonnam = features_to_dataframe(
    region_features["jeonnam"],
    REGION_SPECS["jeonnam"]["label"],
)

# 전라도 = 전북특별자치도 + 전라남도
df_jeolla = pd.concat(
    [df_jeonbuk, df_jeonnam],
    ignore_index=True,
    sort=False,
)

# 전체
df_all = pd.concat(
    [df_seoul, df_gyeonggi, df_jeonbuk, df_jeonnam],
    ignore_index=True,
    sort=False,
)

print("서울:", len(df_seoul))
print("경기:", len(df_gyeonggi))
print("전북:", len(df_jeonbuk))
print("전남:", len(df_jeonnam))
print("전라도 통합:", len(df_jeolla))
print("전체:", len(df_all))

display(df_all.head())


서울: 4
경기: 43
전북: 57
전남: 60
전라도 통합: 117
전체: 164


,fac_nam,fac_tel,cat_nam,fac_o_add,fac_n_add,_feature_id,_geometry_type,region,longitude,latitude
0,서울시립중랑노인전문요양원,02-437-0144,노인복지시설,서울특별시 중랑구 망우동 227-0,서울특별시 중랑구 양원역로 38-0,LT_P_MGPRTFB.420,Point,서울특별시,127.107290,37.604605
1,신내노인요양원,02-490-2600,노인복지시설,서울특별시 중랑구 신내동 642-0,서울특별시 중랑구 신내로 194-0,LT_P_MGPRTFB.446,Point,서울특별시,127.094907,37.616161
2,청운노인복지센터,02-823-3833,노인복지시설,서울특별시 동작구 상도동 290-1,서울특별시 동작구 국사봉1길 145-0,LT_P_MGPRTFB.449,Point,서울특별시,126.933353,37.496020
3,동작실버센터,02-821-8600,노인복지시설,서울특별시 동작구 본동 8-12,서울특별시 동작구 노량진로32길 53-0,LT_P_MGPRTFB.474,Point,서울특별시,126.957156,37.511577
4,예닮원,031-883-9128,노인복지시설,경기도 여주군 능서면 구양리 383-4,경기도 여주군 구양2길 102-0,LT_P_MGPRTFB.385,Point,경기도,127.577661,37.321511



## 6. 중복 확인 및 제거

지역별 공간 필터를 이미 적용했지만, API 데이터 자체에 동일 feature가 중복 저장되어 있을 가능성에 대비해
`_feature_id`가 있으면 이를 기준으로 중복을 제거합니다.


In [7]:

def deduplicate_df(df):
    if df.empty:
        return df.copy()

    if "_feature_id" in df.columns and df["_feature_id"].notna().any():
        return df.drop_duplicates(subset=["_feature_id"]).reset_index(drop=True)

    # feature id가 없으면 전체 행 기준
    return df.drop_duplicates().reset_index(drop=True)


df_seoul = deduplicate_df(df_seoul)
df_gyeonggi = deduplicate_df(df_gyeonggi)
df_jeonbuk = deduplicate_df(df_jeonbuk)
df_jeonnam = deduplicate_df(df_jeonnam)

df_jeolla = pd.concat(
    [df_jeonbuk, df_jeonnam],
    ignore_index=True,
    sort=False,
)
df_jeolla = deduplicate_df(df_jeolla)

df_all = pd.concat(
    [df_seoul, df_gyeonggi, df_jeonbuk, df_jeonnam],
    ignore_index=True,
    sort=False,
)
df_all = deduplicate_df(df_all)

print("중복 제거 후")
print("서울:", len(df_seoul))
print("경기:", len(df_gyeonggi))
print("전북:", len(df_jeonbuk))
print("전남:", len(df_jeonnam))
print("전라도:", len(df_jeolla))
print("전체:", len(df_all))


중복 제거 후
서울: 4
경기: 43
전북: 57
전남: 60
전라도: 117
전체: 164



## 7. DataFrame → CSV / JSON 저장

JSON은 한글이 깨지지 않도록 UTF-8, `force_ascii=False`로 저장합니다.

생성 파일 예시:
- `seoul_elderly_welfare.csv`
- `seoul_elderly_welfare.json`
- `gyeonggi_elderly_welfare.csv`
- `gyeonggi_elderly_welfare.json`
- `jeonbuk_elderly_welfare.csv`
- `jeonbuk_elderly_welfare.json`
- `jeonnam_elderly_welfare.csv`
- `jeonnam_elderly_welfare.json`
- `jeolla_elderly_welfare.json`
- `all_regions_elderly_welfare.json`


In [8]:

DATAFRAMES = {
    "seoul_elderly_welfare": df_seoul,
    "gyeonggi_elderly_welfare": df_gyeonggi,
    "jeonbuk_elderly_welfare": df_jeonbuk,
    "jeonnam_elderly_welfare": df_jeonnam,
    "jeolla_elderly_welfare": df_jeolla,
    "all_regions_elderly_welfare": df_all,
}

saved_files = []

for name, df in DATAFRAMES.items():
    csv_path = OUTPUT_DIR / f"{name}.csv"
    json_path = OUTPUT_DIR / f"{name}.json"

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # records 형태 JSON
    df.to_json(
        json_path,
        orient="records",
        force_ascii=False,
        indent=2,
    )

    saved_files.extend([csv_path, json_path])

print("저장 완료:")
for path in saved_files:
    print(" -", path)


저장 완료:
 - vworld_elderly_welfare_output\seoul_elderly_welfare.csv
 - vworld_elderly_welfare_output\seoul_elderly_welfare.json
 - vworld_elderly_welfare_output\gyeonggi_elderly_welfare.csv
 - vworld_elderly_welfare_output\gyeonggi_elderly_welfare.json
 - vworld_elderly_welfare_output\jeonbuk_elderly_welfare.csv
 - vworld_elderly_welfare_output\jeonbuk_elderly_welfare.json
 - vworld_elderly_welfare_output\jeonnam_elderly_welfare.csv
 - vworld_elderly_welfare_output\jeonnam_elderly_welfare.json
 - vworld_elderly_welfare_output\jeolla_elderly_welfare.csv
 - vworld_elderly_welfare_output\jeolla_elderly_welfare.json
 - vworld_elderly_welfare_output\all_regions_elderly_welfare.csv
 - vworld_elderly_welfare_output\all_regions_elderly_welfare.json



## 8. 원본 GeoJSON feature도 저장

DataFrame으로 변환하기 전의 VWorld feature(`properties + geometry`)를 그대로 보존합니다.
공간 분석이나 지도 시각화를 할 예정이라면 이 파일이 유용합니다.


In [9]:

def save_feature_collection(features, path):
    obj = {
        "type": "FeatureCollection",
        "features": features,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


raw_geojson_paths = {}

for key, features in region_features.items():
    path = OUTPUT_DIR / f"{key}_elderly_welfare_geojson.json"
    save_feature_collection(features, path)
    raw_geojson_paths[key] = path

# 전라도 통합
jeolla_features = (
    region_features["jeonbuk"]
    + region_features["jeonnam"]
)
save_feature_collection(
    jeolla_features,
    OUTPUT_DIR / "jeolla_elderly_welfare_geojson.json",
)

# 전체 통합
all_features = (
    region_features["seoul"]
    + region_features["gyeonggi"]
    + region_features["jeonbuk"]
    + region_features["jeonnam"]
)
save_feature_collection(
    all_features,
    OUTPUT_DIR / "all_regions_elderly_welfare_geojson.json",
)

print("원본 GeoJSON 저장 완료")


원본 GeoJSON 저장 완료



## 9. 결과 점검

시설명/주소 컬럼의 실제 이름은 VWorld 응답 버전에 따라 달라질 수 있으므로,
아래 셀로 전체 컬럼을 먼저 확인합니다.


In [10]:

print("df_all shape:", df_all.shape)
print("\n컬럼 목록:")
print(df_all.columns.tolist())

display(df_all.head(10))

print("\n지역별 건수:")
display(
    df_all["region"]
    .value_counts(dropna=False)
    .rename_axis("region")
    .reset_index(name="count")
)


df_all shape: (164, 10)

컬럼 목록:
['fac_nam', 'fac_tel', 'cat_nam', 'fac_o_add', 'fac_n_add', '_feature_id', '_geometry_type', 'region', 'longitude', 'latitude']


,fac_nam,fac_tel,cat_nam,fac_o_add,fac_n_add,_feature_id,_geometry_type,region,longitude,latitude
0,서울시립중랑노인전문요양원,02-437-0144,노인복지시설,서울특별시 중랑구 망우동 227-0,서울특별시 중랑구 양원역로 38-0,LT_P_MGPRTFB.420,Point,서울특별시,127.107290,37.604605
1,신내노인요양원,02-490-2600,노인복지시설,서울특별시 중랑구 신내동 642-0,서울특별시 중랑구 신내로 194-0,LT_P_MGPRTFB.446,Point,서울특별시,127.094907,37.616161
2,청운노인복지센터,02-823-3833,노인복지시설,서울특별시 동작구 상도동 290-1,서울특별시 동작구 국사봉1길 145-0,LT_P_MGPRTFB.449,Point,서울특별시,126.933353,37.496020
3,동작실버센터,02-821-8600,노인복지시설,서울특별시 동작구 본동 8-12,서울특별시 동작구 노량진로32길 53-0,LT_P_MGPRTFB.474,Point,서울특별시,126.957156,37.511577
4,예닮원,031-883-9128,노인복지시설,경기도 여주군 능서면 구양리 383-4,경기도 여주군 구양2길 102-0,LT_P_MGPRTFB.385,Point,경기도,127.577661,37.321511
5,너싱홈그린힐,031-768-5226,노인복지시설,경기도 광주시 탄벌동 598-4,경기도 광주시 회안대로1583번길 108-0,LT_P_MGPRTFB.386,Point,경기도,127.236000,37.421873
6,예일요양원,031-884-5058,노인복지시설,경기도 여주군 북내면 서원리 186-0,경기도 여주군 서원2길 37-0,LT_P_MGPRTFB.387,Point,경기도,127.697119,37.399173
7,실버릿지그랜드,031-723-5812,노인복지시설,경기도 성남시 수정구 사송동 507-0,경기도 성남시 수정구 사송로80번길 7-0,LT_P_MGPRTFB.388,Point,경기도,127.110203,37.414820
8,남서울너싱홈,031-726-9195,노인복지시설,경기도 광주시 오포읍 신현리 501-0,경기도 광주시 새말길 324-15,LT_P_MGPRTFB.389,Point,경기도,127.176234,37.380608
9,아이너싱홈,031-766-0876,노인복지시설,경기도 광주시 퇴촌면 광동리 279-0,경기도 광주시 산수로 1289-0,LT_P_MGPRTFB.395,Point,경기도,127.306239,37.469603



지역별 건수:


,region,count
0,전라남도,60
1,전북특별자치도,57
2,경기도,43
3,서울특별시,4



## 10. 선택: 광주광역시까지 '전라도/호남'에 포함하고 싶은 경우

현재 `df_jeolla`는 **전북특별자치도 + 전라남도**만 포함합니다.

광주광역시까지 포함하려면 `REGION_SPECS`에 아래 항목을 추가한 뒤,
4번 셀부터 다시 실행하면 됩니다.

```python
"gwangju": {
    "label": "광주광역시",
    "aliases": ["광주광역시", "광주"],
}
```

그 후:

```python
df_gwangju = features_to_dataframe(
    region_features["gwangju"],
    REGION_SPECS["gwangju"]["label"],
)

df_honam = pd.concat(
    [df_jeonbuk, df_jeonnam, df_gwangju],
    ignore_index=True,
    sort=False,
)
```



## 오류가 날 때 체크

### `INVALID_KEY`, 인증 관련 오류
- VWorld API 키가 정확한지 확인
- 키에 등록한 도메인과 `VWORLD_DOMAIN`이 일치하는지 확인

### `NOT_FOUND`
- 해당 범위에 데이터가 없거나
- 서비스 ID/필터 조건을 다시 확인해야 할 수 있음

### 전북 경계를 못 찾는 경우
VWorld 경계 데이터가 과거 명칭인 `전라북도`를 사용할 수 있습니다.
이 노트북은 두 명칭을 모두 처리하도록 구성되어 있습니다.

### 데이터가 예상보다 적은 경우
노인복지시설 데이터 자체의 갱신 시점/제공 범위가 제한적일 수 있습니다.
이 노트북은 VWorld에서 제공되는 `LT_P_MGPRTFB` 레이어를 그대로 수집합니다.
